In [2]:
from tempfile import gettempdir
import biotite.structure as struc
import biotite.structure.io as strucio
import biotite.database.rcsb as rcsb
import numpy as np

# Explore Biotite: Example `biotite` usage for reference
Includes standard distance computations using the Biotite library. 
For more information, see the biotite [tutorial](https://www.biotite-python.org/latest/tutorial/index.html)

### Standard `AtomArray` filtering

In [3]:
# Load the file
file_path = rcsb.fetch("1fu2", "bcif", gettempdir())
array = strucio.load_structure(file_path)  # Only one model
print(array.shape)

(1630,)


In [10]:
# Get the coordinates of all of the CA atoms
ca = array[array.atom_name == "CA"]
print(ca.coord.shape)

# Get the coordinates of chain A
chain = array[array.chain_id == "A"]
print(chain.coord.shape)

#  Get all unique chain ID's
chain_ids = np.unique(array.chain_id)

(204, 3)
(163, 3)


For chain-level and residue-level operations (e.g., loop through chains, loop through molecules, etc.), see the [documentation](https://www.biotite-python.org/apidoc/biotite.structure.html#module-biotite.structure)

### `struc.distance` — Calculate atom-to-atom, array-to-atom, calculate array-to-array (pairwise) distances

Note: For more efficient computation of finding atoms within a cutoff distance, look at `CellList`

In [34]:
file_path = rcsb.fetch("1l2y", "bcif", gettempdir())
stack = strucio.load_structure(file_path)
stack = stack[:, stack.atom_name == "CA"]  # Select only C-alpha atoms
array = stack[0]  # Get the first model (of the NMR ensemble of 38)

### ATOM TO ATOM
# Distance between two atoms
distance = struc.distance(array.coord[0], array.coord[1])  # Distance between first two C-alpha atoms
print(f"Distance between first two C-alpha atoms: {distance:.2f} Å")

### ARRAY TO ATOM
# Distance between one atom and all other atoms in the array
distance = struc.distance(array[0], array)
print(f"Distances between first C-alpha atom and all other C-alpha atoms: {distance}")
# If we want to get what is the closest omintom to the first atom
min_distance = np.min(distance[1:])  # The first element is the distance to itself
mask = distance == min_distance
closest_atom = array[mask]
print(f"Closest atom to the first C-alpha atom: {closest_atom}, with distance {min_distance:.2f} Å")

# ARRAY TO ARRAY
# Pairwise distances between NMR model c-alpha atoms
pairwise_distance = struc.distance(stack[0], stack[1])
print(f"Pairwise distances between first and second NMR model C-alpha atoms: {pairwise_distance}")

# ADJACENT ATOMS
array = stack[0]  # Conside only the first model
# Distances between two adjacent CA
adjacent_distance = struc.distance(array[:-1], array[1:])
print(f"Distances between adjacent C-alpha atoms: {adjacent_distance}")

Distance between first two C-alpha atoms: 3.88 Å
Distances between first C-alpha atom and all other C-alpha atoms: [ 0.         3.876399   5.5766597  5.038891   6.316409   8.766815
  9.908135  10.614817  12.890331  14.806679  13.5011635 16.87541
 18.723566  17.224289  19.111933  16.193     15.514756  12.377812
 10.445933  12.058967 ]
Closest atom to the first C-alpha atom:     A       2  LEU CA     C        -4.923    4.002   -2.452, with distance 3.88 Å
Pairwise distances between first and second NMR model C-alpha atoms: [3.4344199  0.37241507 0.22178602 0.10823133 0.15207201 0.17017055
 0.22572783 0.47650498 0.29493213 0.1548351  0.2832347  0.40683934
 0.1355508  0.3676874  0.46464393 0.57544243 0.33707416 0.25703317
 0.34762198 0.38818687]
Distances between adjacent C-alpha atoms: [3.876399  3.8605015 3.87147   3.8455799 3.8666048 3.8585181 3.8818028
 3.860987  3.8909183 3.8635552 3.8862703 3.8756127 3.8746684 3.8655493
 3.8662775 3.8776622 3.8603828 3.8582468 3.8642192]


### `CellList` — Efficient computation of distance-bounded lookups

In [41]:
file_path = rcsb.fetch("1fu2", "bcif", gettempdir())  # Oligosaccharide
array = strucio.load_structure(file_path)  # Only one model
print(array.shape)

(1630,)


In [72]:
# Build CellList
cell_list = struc.CellList(array, cell_size=5.0)

### ATOMS NEAR ATOM
# Get all atoms within 5 Å of the first atom
print(f"Target atom: {array[0].coord} {array[0].atom_name} {array[0].res_name} {array[0].res_id}")
near_atoms = cell_list.get_atoms(array[0].coord, radius=4)
print(f"Number of atoms within 7 Å of the first atom: {near_atoms.shape[0]}")
print(f"Atom indices: {near_atoms}")
chain_ids = array.chain_id[near_atoms]
print(f"Chain IDs: {chain_ids}")
residue_ids = array.res_id[near_atoms]
print(f"Residue IDs: {residue_ids}")
print(f"Residue names: {array.res_name[near_atoms]}")

### ATOMS NEAR ATOM ARRAY (CHAIN)
# Get all atoms within 5 Å of any atom in chain A
chain_a_atoms = array[array.chain_id == "A"]
near_atoms = cell_list.get_atoms(chain_a_atoms.coord, radius=5)
print(f"Number of atoms within 5 Å of chain A: {len(np.unique(near_atoms.ravel())) - 1}")
print(f"Shape of atom indices: {near_atoms.shape}")
print(f"Atom indices: {near_atoms[:2]}")  # -1 is used for padding

# We can also return directly as a mask rather than as indices
near_atom_mask = cell_list.get_atoms(chain_a_atoms.coord, radius=5, as_mask=True)
print(f"Number of atoms in the atom array: {len(array)}")
print(f"Shape of mask: {near_atom_mask.shape}")
print(f"Mask: {near_atom_mask[:2]}")
# Collapse mask
collapsed_mask = np.any(near_atom_mask, axis=0)
print(f"Number of atoms within 5 Å of chain A: {np.sum(collapsed_mask)}")
near_atoms = array[collapsed_mask]
# Filter out atoms within chain A
near_atoms = near_atoms[near_atoms.chain_id != "A"]
print(f"Number of atoms within 5 Å of chain A (excluding chain A atoms): {len(near_atoms)}")

Target atom: [  3.124  19.323 -14.539] N GLY 1
Number of atoms within 7 Å of the first atom: 9
Atom indices: [  0 399 400   2   4 398 404   1   3]
Chain IDs: ['A' 'B' 'B' 'A' 'A' 'B' 'B' 'A' 'A']
Residue IDs: [ 1 30 30  1  2 30 30  1  1]
Residue names: ['GLY' 'THR' 'THR' 'GLY' 'ILE' 'THR' 'THR' 'GLY' 'GLY']
Number of atoms within 5 Å of chain A: 258
Shape of atom indices: (163, 42)
Atom indices: [[391 397  20   0  23  24 399 400 401 403   2   4   5   6  12  19 398 404
    1   3  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1
   -1  -1  -1  -1  -1  -1]
 [  0 399 400   2   4   5   6  12 398 404   8   1   3 147  -1  -1  -1  -1
   -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1
   -1  -1  -1  -1  -1  -1]]
Number of atoms in the atom array: 1630
Shape of mask: (163, 1630)
Mask: [[ True  True  True ... False False False]
 [ True  True  True ... False False False]]
Number of atoms within 5 Å of chain A: 258
Number of atoms within 5 Å of chain A (excludin